# Palette + Pixelation Pipeline – Batch Version
Reads all images from `test_cases/`, estimates optimal block size (normal and fast), extracts palette via LAB + K‑means, pixelates the image, and saves both results to `output/`.

A summary table is printed with: **ImageName | Method | GridSize | ColorsPicked | ElapsedTime**.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
%matplotlib inline

# Import the external modules (assumed to be in the same directory)
from smart_grid import estimate_block_size_fast, estimate_block_size
from palette import derive_palette
from derive_grid import pixelate_image, derive_block_color
from sklearn.cluster import KMeans

In [ ]:
# ===== CONFIGURATION =====
INPUT_DIR = Path(r"D:\pythonProject\Private\CUHKSZ Schoolwork\Y1T2S\ECE4512\Pixelate Everything\test_cases")   # <-- change as needed
OUTPUT_DIR = Path(r"D:\pythonProject\Private\CUHKSZ Schoolwork\Y1T2S\ECE4512\Pixelate Everything\MVP ver.3\output")  # <-- change as needed
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')
# =========================

# Ensure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check input directory
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input folder not found: {INPUT_DIR.resolve()}")

In [ ]:
# ===== ORIGINAL FUNCTIONS FROM main.py (KEPT UNCHANGED) =====

# PATH = r"D:\pythonProject\Private\CUHKSZ Schoolwork\Y1T2S\ECE4512\Pixelate Everything\test_cases"
# OUT_PATH = r"D:\pythonProject\Private\CUHKSZ Schoolwork\Y1T2S\ECE4512\Pixelate Everything\MVP ver.3\output"

def _batch_process(input_dir: str, output_dir: str):
    import os
    from pathlib import Path

    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Supported image formats
    image_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')

    for img_file in input_path.iterdir():
        if not img_file.is_file():
            continue
        if img_file.suffix.lower() not in image_exts:
            continue

        print(f"Processing {img_file.name} …")

        pixelated, pixelated_fast = _process_image(str(img_file))

        # Save results
        stem = img_file.stem
        cv2.imwrite(str(output_path / f"{stem}_normal.png"), pixelated)
        cv2.imwrite(str(output_path / f"{stem}_fast.png"),   pixelated_fast)

    print("Batch processing finished.")


def _process_image(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR_BGR)
    if img is None:
        print(f"Error: Could not read image from {path}")
        exit(1)
    recommended_size = estimate_block_size(img)
    recommended_size_fast = estimate_block_size_fast(img)
    color_map, colors = derive_palette(img)
    pixelated = pixelate_image(img, recommended_size, color_map, colors)
    pixelated_fast = pixelate_image(img, recommended_size_fast, color_map, colors)
    return pixelated, pixelated_fast

In [ ]:
# ===== BATCH PROCESSING WITH TIMING AND TABLE =====

# We will reuse the logic but add timing and collect metrics

def process_with_timing(input_dir: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Prepare a list to hold table rows
    table_rows = []
    
    for img_path in sorted(input_dir.iterdir()):
        if not img_path.is_file():
            continue
        if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        
        print(f"Processing {img_path.name} …")
        
        # Read image
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img is None:
            print(f"⚠️  Could not read: {img_path.name}")
            continue
        
        # Estimate block sizes
        recommended_size = estimate_block_size(img)
        recommended_size_fast = estimate_block_size_fast(img)
        
        # Derive palette (this gives the number of colors)
        color_map, colors = derive_palette(img)
        num_colors = len(colors)
        
        # ---- Normal method ----
        start = time.time()
        pixelated = pixelate_image(img, recommended_size, color_map, colors)
        elapsed_normal = time.time() - start
        
        # ---- Fast method ----
        start = time.time()
        pixelated_fast = pixelate_image(img, recommended_size_fast, color_map, colors)
        elapsed_fast = time.time() - start
        
        # Save images
        stem = img_path.stem
        cv2.imwrite(str(output_dir / f"{stem}_normal.png"), pixelated)
        cv2.imwrite(str(output_dir / f"{stem}_fast.png"), pixelated_fast)
        
        # Collect table rows
        table_rows.append({
            "ImageName": img_path.name,
            "Method": "normal",
            "GridSize": recommended_size,
            "ColorsPicked": num_colors,
            "ElapsedTime": elapsed_normal
        })
        table_rows.append({
            "ImageName": img_path.name,
            "Method": "fast",
            "GridSize": recommended_size_fast,
            "ColorsPicked": num_colors,
            "ElapsedTime": elapsed_fast
        })
    
    # Print the table
    print("\n" + "="*80)
    print(f"{'ImageName':<30} | {'Method':<8} | {'GridSize':<10} | {'ColorsPicked':<14} | {'ElapsedTime (s)':<15}")
    print("-"*80)
    for row in table_rows:
        print(f"{row['ImageName']:<30} | {row['Method']:<8} | {row['GridSize']:<10} | {row['ColorsPicked']:<14} | {row['ElapsedTime']:<15.4f}")
    print("="*80)
    print(f"\nDone. All images saved to {output_dir.resolve()}")

# Run the batch processing with timing
process_with_timing(INPUT_DIR, OUTPUT_DIR)

In [ ]:
# ===== OPTIONAL: SHOW COMPARISONS =====
SHOW_COMPARISONS = True

if SHOW_COMPARISONS:
    for img_path in sorted(INPUT_DIR.iterdir()):
        if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        
        normal_path = OUTPUT_DIR / f"{img_path.stem}_normal.png"
        fast_path = OUTPUT_DIR / f"{img_path.stem}_fast.png"
        
        if not normal_path.exists() or not fast_path.exists():
            continue
        
        orig_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        normal_bgr = cv2.imread(str(normal_path), cv2.IMREAD_COLOR)
        fast_bgr = cv2.imread(str(fast_path), cv2.IMREAD_COLOR)
        if orig_bgr is None or normal_bgr is None or fast_bgr is None:
            continue
        
        # BGR -> RGB for matplotlib
        orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
        normal_rgb = cv2.cvtColor(normal_bgr, cv2.COLOR_BGR2RGB)
        fast_rgb = cv2.cvtColor(fast_bgr, cv2.COLOR_BGR2RGB)
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(orig_rgb)
        axes[0].set_title(f"Original: {img_path.name}")
        axes[0].axis("off")
        axes[1].imshow(normal_rgb)
        axes[1].set_title("Normal pixelation")
        axes[1].axis("off")
        axes[2].imshow(fast_rgb)
        axes[2].set_title("Fast pixelation")
        axes[2].axis("off")
        plt.tight_layout()
        plt.show()